# Sample: fuentes públicas, muestreo y trazabilidad

VulnMiner · DavidPilaguano · SEMMA con datos públicos verificables.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / 'data/dataset.csv').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image
df = pd.read_csv(ROOT / 'data/dataset.csv', keep_default_na=False)
df['labels'] = df.labels.map(json.loads)
manifest = json.loads((ROOT / 'data/public_manifest.json').read_text(encoding='utf-8'))
results = json.loads((ROOT / 'output/evaluation/metrics.json').read_text(encoding='utf-8'))
selected = results['candidates'][results['selected']]


Se usan dos suites públicas con oráculos oficiales: OWASP BenchmarkPython y BenchmarkJava. Son programas de prueba controlados, no código de incidentes de producción. Se conserva un suplemento original para categorías API que esos oráculos no etiquetan.

In [2]:
display(pd.DataFrame(manifest['sources'])[['name','language','commit','license','records_downloaded']])
display(df.groupby(['source','split']).size().unstack(fill_value=0))
print('Casos descargados:', manifest['raw_public_records'])
print('Duplicados excluidos:', manifest['duplicate_records_removed'])
print('Total preparado:', len(df))

,name,language,commit,license,records_downloaded
0,BenchmarkPython,Python,f1291485808b66e20ddb6b01b10dc71b3df8c8ba,GPL-3.0-only,1230
1,BenchmarkJava,Java,10dfcc54ad02dcfcb358b0ba4a0fef8c919a6d17,GPL-2.0-only,2740


split,test,train,validation
source,,,
BenchmarkJava,389,1953,394
BenchmarkPython,178,879,173
original-controlled-v1,168,672,168


Casos descargados: 3970
Duplicados excluidos: 4
Total preparado: 4974


In [3]:
import hashlib
for source in manifest['sources']:
    oracle = ROOT / 'data/public' / source['name'] / source['oracle']
    assert hashlib.sha256(oracle.read_bytes()).hexdigest() == source['expectedresults_sha256']
assert not df.sha256.duplicated().any()
assert df.groupby('project').split.nunique().max() == 1
public = df[df.source != 'original-controlled-v1']
assert not public.code.str.contains(r'BenchmarkTest\d+').any()
print('Oráculos verificados, sin duplicados exactos ni grupos repartidos.')

Oráculos verificados, sin duplicados exactos ni grupos repartidos.


Las particiones públicas se obtienen mediante siete folds estratificados por fuente, escenario y estado, agrupando por huella estructural sin identificadores ni literales. Se reservan dos folds para validación y prueba. Las familias propias conservan su separación original. Esto reduce fuga por plantillas, pero no prueba independencia semántica completa ni separación por repositorio: ambos Benchmark participan en las tres particiones.